# 02 — Tạo đặc trưng có thể biết tại từng cutoff

**Câu hỏi:** Nếu chỉ biết tổng click, liệu model có phân biệt được sinh viên học đều với sinh viên đang dừng học? Notebook tạo thêm tỷ lệ và cửa sổ hoạt động gần ngày dự báo. Dữ liệu đầu vào vẫn là **7 bảng gốc**, không lấy một bảng cuối kỳ rồi cắt ngang hàng.

In [ ]:
from pathlib import Path
import sys
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / 'research_code').is_dir() and (candidate / 'shared_code').is_dir()), None)
assert ROOT is not None, 'Open Jupyter from packages/ml or its notebooks directory.'
sys.path.insert(0, str(ROOT))
from shared_code.thuc_nghiem import (
    GOC, NGHIEN_CUU, DATA, KET_QUA, HINH, CAC_MOC,
    tao_thu_muc, duong_dan, chia_nhom, nap_du_lieu, chon_dac_trung,
    tien_xu_ly, bo_sung_ty_le, bo_sung_cua_so_vle,
)
tao_thu_muc()
print('Dữ liệu nguồn:', DATA)
print('Kết quả riêng:', KET_QUA)

In [ ]:
import numpy as np
import pandas as pd
from research_code.data import create_asof
from IPython.display import display

source_ids = pd.read_csv(DATA / 'studentInfo.csv', usecols=['id_student']).id_student
split = chia_nhom(source_ids, seed=41)
split.to_csv(KET_QUA / 'phan_tap_sinh_vien.csv', index=False, encoding='utf-8-sig')
display(split.split.value_counts().rename('số sinh viên').to_frame())
assert split.id_student.is_unique

## 1. Cắt sự kiện trước, tạo feature sau

`create_asof` gộp bảy nguồn theo khóa enrollment, tính ngày cutoff cho từng môn, bỏ sự kiện sau ngày đó. Sau đó tính thêm `clicks_per_day`, `active_day_ratio`, `submission_rate`, `days_since_last_activity` và click trong 7/30 ngày cuối. Ở các khoảng thời gian ngắn không có hoạt động, số click được đặt bằng 0 chứ không dùng thông tin sau cutoff.

In [ ]:
summary = []
for cutoff in CAC_MOC:
    existing = duong_dan(cutoff)
    if existing.exists():
        asof = nap_du_lieu(cutoff)
    else:
        asof = create_asof(DATA, cutoff, cutoff_mode='fraction', score_policy='strict')
    if {'click_trend_ratio', 'split'} <= set(asof.columns):
        engineered = asof
    else:
        engineered = bo_sung_cua_so_vle(bo_sung_ty_le(asof))
        engineered = engineered.merge(split, on='id_student', how='left', validate='many_to_one')
    assert engineered.split.notna().all()
    assert not engineered[['id_student','code_module','code_presentation']].duplicated().any()
    assert engineered.last_activity_day.dropna().le(engineered.loc[engineered.last_activity_day.notna(),'cutoff_day']).all()
    assert (engineered[['clicks_recent_7', 'clicks_recent_30', 'clicks_first_half', 'clicks_second_half']] >= 0).all().all()
    engineered.to_csv(existing, index=False, encoding='utf-8-sig')
    summary.append({'cutoff': cutoff, 'rows': len(engineered), 'features': len(chon_dac_trung(engineered)),
                    'risk_rate_train': engineered.loc[engineered.split == 'train', 'risk'].mean(),
                    **engineered.split.value_counts().to_dict()})
    print(f'{cutoff:.0%}: {len(engineered):,} enrollment')
summary = pd.DataFrame(summary)
summary.to_csv(KET_QUA/'tong_quan_cac_moc.csv', index=False, encoding='utf-8-sig')
display(summary.round(4))

## 2. Đọc một vài dòng: chúng ta đã biết gì?

`clicks_recent_7` thể hiện mức hoạt động sát cutoff; `click_trend_ratio` so sánh nửa sau với nửa đầu của khoảng đã quan sát. Giá trị lớn không mặc nhiên là học tốt: chỉ model và validation mới cho biết nó có giúp nhận diện nguy cơ hay không.

In [ ]:
columns = ['code_module','code_presentation','id_student','cutoff_day','total_clicks_to_t',
           'clicks_recent_7','clicks_recent_30','click_trend_ratio','submission_rate','risk','split']
display(nap_du_lieu(0.5)[columns].head(3).assign(id_student='ẩn'))

**Lưu ý nghiên cứu:** Ba dòng hiển thị đã che `id_student` trong bản công bố; khi chạy lại, ID cũng được che trong ô hiển thị. Dữ liệu enrollment chỉ lưu cục bộ; `split` là nhãn quản lý đánh giá, tuyệt đối không đưa vào feature.